In [4]:
# Importing required libraries

import requests
import pandas as pd
import json
import time
import os
print("Libraries imported successfully!")

Libraries imported successfully!


In [5]:
# SEC company ticker database

url = "https://www.sec.gov/files/company_tickers.json"

headers = {
    "User-Agent": "Samara sa510633@ucf.edu"
}

response = requests.get(url, headers=headers)

print("Status Code:", response.status_code)

Status Code: 200


In [6]:
#Creating the variables needed

url = "https://www.sec.gov/files/company_tickers.json"

response = requests.get(url, headers=headers)

print("Status Code:", response.status_code)

company_data = response.json()

companies = pd.DataFrame(company_data.values())

companies["cik_str"] = companies["cik_str"].astype(str).str.zfill(10)

display(companies.head())

Status Code: 200


,cik_str,ticker,title
0,0001045810,NVDA,NVIDIA CORP
1,0000320193,AAPL,Apple Inc.
2,0000789019,MSFT,MICROSOFT CORP
3,0001652044,GOOGL,Alphabet Inc.
4,0001018724,AMZN,AMAZON COM INC


In [7]:
# Technology companies for assignment

selected_tickers = [
    "NVDA",
    "AAPL",
    "MSFT",
    "AMZN",
    "META",
    "TSLA",
    "AMD",
    "ADBE"
]

selected_companies = companies[
    companies["ticker"].isin(selected_tickers)
].copy()

print(selected_companies[
    ["title", "ticker", "cik_str"]
])

                          title ticker     cik_str
0                   NVIDIA CORP   NVDA  0001045810
1                    Apple Inc.   AAPL  0000320193
2                MICROSOFT CORP   MSFT  0000789019
4                AMAZON COM INC   AMZN  0001018724
5          Meta Platforms, Inc.   META  0001326801
7                   Tesla, Inc.   TSLA  0001318605
13   ADVANCED MICRO DEVICES INC    AMD  0000002488
173                  ADOBE INC.   ADBE  0000796343


In [8]:
#Retrieving the filing records
all_filings = []

for _, company in selected_companies.iterrows():

    cik = company["cik_str"]
    ticker = company["ticker"]
    company_name = company["title"]

    submissions_url = f"https://data.sec.gov/submissions/CIK{cik}.json"

    response = requests.get(submissions_url, headers=headers)

    if response.status_code == 200:

        filing_data = response.json()

        recent_filings = pd.DataFrame(
            filing_data["filings"]["recent"]
        )

        eight_k = recent_filings[
            recent_filings["form"] == "8-K"
        ].copy()

        eight_k["company_name"] = company_name
        eight_k["ticker"] = ticker
        eight_k["cik"] = cik

        all_filings.append(eight_k)

        print(ticker, ":", len(eight_k), "8-K filings found")

    else:
        print("Error:", ticker, response.status_code)

    time.sleep(0.2)

NVDA : 60 8-K filings found
AAPL : 101 8-K filings found
MSFT : 59 8-K filings found
AMZN : 63 8-K filings found
META : 24 8-K filings found
TSLA : 128 8-K filings found
AMD : 110 8-K filings found
ADBE : 81 8-K filings found


In [9]:
#One dataset and save metadata
if all_filings:

    filings_df = pd.concat(all_filings, ignore_index=True)

    filings_df = filings_df.sort_values(
        "filingDate",
        ascending=False
    ).head(100)

    print("Total filings selected:", len(filings_df))

    display(
        filings_df[
            ["company_name", "ticker", "filingDate", "form"]
        ].head(10)
    )

    filings_df.to_csv("data/sec_8k_filings.csv", index=False)

    print("Filing records saved!")

else:
    print("No filings retrieved. Check the errors above.")

Total filings selected: 100


,company_name,ticker,filingDate,form
161,MICROSOFT CORP,MSFT,2026-11-04,8-K
435,ADVANCED MICRO DEVICES INC,AMD,2026-11-03,8-K
60,Apple Inc.,AAPL,2026-11-02,8-K
220,AMAZON COM INC,AMZN,2026-10-29,8-K
283,"Meta Platforms, Inc.",META,2026-10-29,8-K
162,MICROSOFT CORP,MSFT,2026-10-28,8-K
163,MICROSOFT CORP,MSFT,2026-10-28,8-K
307,"Tesla, Inc.",TSLA,2026-10-21,8-K
284,"Meta Platforms, Inc.",META,2026-10-21,8-K
308,"Tesla, Inc.",TSLA,2026-10-02,8-K


Filing records saved!


In [10]:
#importing beautifulsoup

from bs4 import BeautifulSoup

In [11]:
#Downloading the filings
def get_filing_text(row):

    cik = str(int(row["cik"]))
    accession = row["accessionNumber"].replace("-", "")
    document = row["primaryDocument"]

    filing_url = (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{cik}/{accession}/{document}"
    )

    response = requests.get(
        filing_url,
        headers=headers,
        timeout=30
    )

    if response.status_code != 200:
        print("Download failed:", response.status_code)
        return ""

    soup = BeautifulSoup(response.content, "html.parser")

    for tag in soup(["script", "style"]):
        tag.decompose()

    filing_text = soup.get_text(" ", strip=True)

    return " ".join(filing_text.split())

In [12]:
#Testing the filing
sample_text = get_filing_text(filings_df.iloc[0])

print("Characters downloaded:", len(sample_text))
print(sample_text[:1500])

Characters downloaded: 8837
8-K false 0000789019 0000789019 2026-10-29 2026-10-29 0000789019 msft:NotesTwoPointOneTwoFivePercentDueDecemberSixTwentyTwentyOneMember 2026-10-29 2026-10-29 0000789019 msft:NotesThreePointOneTwoFivePercentDueDecemberSixTwentyTwentyEightMember 2026-10-29 2026-10-29 0000789019 msft:NotesTwoPointSixTwoFivePercentDueMayTwoTwentyThirtyThreeMember 2026-10-29 2026-10-29 UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 FORM 8-K CURRENT REPORT Pursuant to Section 13 or 15(d) of the Securities Exchange Act of 1934 Date of Report (date of earliest event reported): October 29, 2026 MICROSOFT CORPORATION (Exact name of registrant as specified in its charter) Washington 001-37845 91-1144442 (State or other jurisdiction of incorporation) (Commission File Number) (IRS Employer Identification No.) One Microsoft Way , Redmond , Washington 98052-6399 (Address of principal executive offices) (Zip Code) ( 425 ) 882-8080 (Registrant’s telephone number, inc

In [13]:
#Dowloading 20 batch file
filings_sample = filings_df.head(20).copy()

filing_texts = []

for index, row in filings_sample.iterrows():

    try:
        text = get_filing_text(row)
    except requests.RequestException as error:
        print("Request error:", error)
        text = ""

    filing_texts.append(text)

    print(
        row["ticker"],
        "-",
        len(text),
        "characters downloaded"
    )

    time.sleep(0.2)

filings_sample["filing_text"] = filing_texts

print("Successfully downloaded:",
      (filings_sample["filing_text"].str.len() > 0).sum())

MSFT - 8837 characters downloaded
AMD - 4391 characters downloaded
AAPL - 3987 characters downloaded
AMZN - 5032 characters downloaded
META - 23335 characters downloaded
MSFT - 37718 characters downloaded
MSFT - 3581 characters downloaded
TSLA - 3240 characters downloaded
META - 19751 characters downloaded
TSLA - 3055 characters downloaded
TSLA - 9850 characters downloaded
AMD - 3755 characters downloaded
AMZN - 6712 characters downloaded
ADBE - 14289 characters downloaded
AMZN - 5613 characters downloaded
ADBE - 5580 characters downloaded
NVDA - 8074 characters downloaded
MSFT - 4188 characters downloaded
NVDA - 3988 characters downloaded
AMD - 5016 characters downloaded
Successfully downloaded: 20


In [14]:
#Saved dataset
filings_sample.to_csv(
    "data/sec_8k_filings_with_text.csv",
    index=False
)

print("Filing texts saved!")

Filing texts saved!


In [15]:
#connecting llama
response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.2",
        "prompt": "Say hello in one sentence.",
        "stream": False
    },
    timeout=120
)

response.raise_for_status()
print(response.json()["response"])

Hello!


In [16]:
#Testing the Ollama
response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.2",
        "prompt": "What is an SEC Form 8-K?.",
        "stream": False
    },
    timeout=120
)

response.raise_for_status()

print(response.json()["response"])

An SEC Form 8-K is a quarterly and current report filed by publicly traded companies with the United States Securities and Exchange Commission (SEC). It is a mandatory filing that provides a snapshot of a company's current situation, highlighting any significant events, transactions, or changes that may impact investors.

Form 8-K reports are typically filed immediately after an event has occurred, and they are used to notify the public and the SEC of material information that may affect the company's stock price, financial condition, or operations.

Some common examples of events that may trigger an SEC Form 8-K filing include:

1. Merger or acquisition announcements
2. Stock option grants or changes
3. Insider sales or purchases
4. Changes in executive leadership or board membership
5. Bankruptcy or financial distress
6. Changes in financial condition or results of operations
7. Material contract or agreement changes
8. Dismissal of lawsuits or settlements
9. Amendments to the compan

In [17]:
#extracting the products using llama
def extract_products(filing_text):

    prompt = f"""
    Analyze this SEC Form 8-K filing.

    Identify ONLY newly announced commercial products,
    software, hardware, platforms, or services.

    Do NOT classify the following as new products:
    - Common stock or preferred stock
    - Depositary shares
    - Bonds or debt securities
    - Dividends or stock offerings
    - Existing products
    - Financial instruments

    Only identify a product if the filing explicitly
    announces its introduction or launch.

    Return valid JSON in this format:
    {{
        "products": [
            {{
                "new_product": "Product name",
                "product_description": "Description under 180 characters"
            }}
        ]
    }}

    If no qualifying products are announced, return:
    {{"products": []}}

    Filing text:
    {filing_text[:12000]}
    """

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3.2",
            "prompt": prompt,
            "format": "json",
            "stream": False
        },
        timeout=300
    )

    response.raise_for_status()

    return json.loads(response.json()["response"])

In [18]:
#Testing the files
results = []

for _, row in filings_sample.iterrows():

    print("Analyzing:", row["ticker"])

    if not row["filing_text"]:
        continue

    try:
        extracted = extract_products(row["filing_text"])

        for product in extracted.get("products", []):
            results.append({
                "company_name": row["company_name"],
                "stock_name": row["ticker"],
                "filing_time": row["filingDate"],
                "new_product": product["new_product"],
                "product_description":
                    product["product_description"][:179]
            })

    except Exception as error:
        print("Error:", error)

print("Products found:", len(results))

Analyzing: MSFT
Analyzing: AMD
Analyzing: AAPL
Analyzing: AMZN
Analyzing: META
Analyzing: MSFT
Analyzing: MSFT
Analyzing: TSLA
Analyzing: META
Analyzing: TSLA
Analyzing: TSLA
Analyzing: AMD
Analyzing: AMZN
Analyzing: ADBE
Analyzing: AMZN
Analyzing: ADBE
Analyzing: NVDA
Analyzing: MSFT
Analyzing: NVDA
Analyzing: AMD
Products found: 48


In [3]:
#CSV with structured columns
# Create and clean final CSV
import os
import pandas as pd

columns = [
    "company_name",
    "stock_name",
    "filing_time",
    "new_product",
    "product_description"
]

final_df = pd.read_csv("data/extracted_products.csv")

excluded_terms = [
    "notes", "bonds", "stock", "shares",
    "debt", "merger", "acquisition",
    "leadership", "investor metrics",
    "not specified", "financial results"
]

pattern = "|".join(excluded_terms)

final_df = final_df[
    ~final_df["new_product"].fillna("").str.lower().str.contains(
        pattern, regex=True
    )
].copy()

final_df["product_description"] = (
    final_df["product_description"].fillna("").astype(str).str[:179]
)

os.makedirs("data", exist_ok=True)

final_df.to_csv("data/extracted_products.csv", index=False)

print("Final products:", len(final_df))
print("CSV saved successfully!")

Final products: 6
CSV saved successfully!


In [20]:
#checking for products
print("Products extracted:", len(final_df))

Products extracted: 47


In [21]:
display(final_df)

,company_name,stock_name,filing_time,new_product,product_description
0,MICROSOFT CORP,MSFT,2026-11-04,Floating Rate Notes due 2028,Floating Rate Notes due 2028
1,MICROSOFT CORP,MSFT,2026-11-04,Floating Rate Notes due 2029,Floating Rate Notes due 2029
2,MICROSOFT CORP,MSFT,2026-11-04,5.000% Notes due 2029,5.000% Notes due 2029
3,MICROSOFT CORP,MSFT,2026-11-04,5.200% Notes due 2031,5.200% Notes due 2031
4,MICROSOFT CORP,MSFT,2026-11-04,5.350% Notes due 2033,5.350% Notes due 2033
5,MICROSOFT CORP,MSFT,2026-11-04,5.550% Notes due 2036,5.550% Notes due 2036
6,MICROSOFT CORP,MSFT,2026-11-04,6.150% Notes due 2046,6.150% Notes due 2046
7,MICROSOFT CORP,MSFT,2026-11-04,6.250% Notes due 2056,6.250% Notes due 2056
8,MICROSOFT CORP,MSFT,2026-11-04,6.350% Notes due 2066,6.350% Notes due 2066
9,ADVANCED MICRO DEVICES INC,AMD,2026-11-03,AMD Ryzen 7000 Series,New Ryzen 7000 series processors


In [22]:
#Checking the description length
final_df["description_length"] = (
    final_df["product_description"].str.len()
)

display(final_df[[
    "new_product",
    "product_description",
    "description_length"
]])

,new_product,product_description,description_length
0,Floating Rate Notes due 2028,Floating Rate Notes due 2028,28
1,Floating Rate Notes due 2029,Floating Rate Notes due 2029,28
2,5.000% Notes due 2029,5.000% Notes due 2029,21
3,5.200% Notes due 2031,5.200% Notes due 2031,21
4,5.350% Notes due 2033,5.350% Notes due 2033,21
5,5.550% Notes due 2036,5.550% Notes due 2036,21
6,6.150% Notes due 2046,6.150% Notes due 2046,21
7,6.250% Notes due 2056,6.250% Notes due 2056,21
8,6.350% Notes due 2066,6.350% Notes due 2066,21
9,AMD Ryzen 7000 Series,New Ryzen 7000 series processors,32


In [23]:
excluded_terms = [
    "common stock",
    "preferred stock",
    "depositary shares",
    "debt securities",
    "senior notes",
    "floating rate notes",
    "fixed rate notes",
    "financial results",
    "third quarter",
    "radeon pro 680m",
    "epyc 7003",
    "ryzen 7000"
]

final_df = final_df[
    ~final_df["new_product"].str.lower().str.contains(
        "|".join(excluded_terms),
        na=False,
        regex=True
    )
].copy()

display(final_df)

print("Remaining candidates:", len(final_df))

,company_name,stock_name,filing_time,new_product,product_description,description_length
2,MICROSOFT CORP,MSFT,2026-11-04,5.000% Notes due 2029,5.000% Notes due 2029,21
3,MICROSOFT CORP,MSFT,2026-11-04,5.200% Notes due 2031,5.200% Notes due 2031,21
4,MICROSOFT CORP,MSFT,2026-11-04,5.350% Notes due 2033,5.350% Notes due 2033,21
5,MICROSOFT CORP,MSFT,2026-11-04,5.550% Notes due 2036,5.550% Notes due 2036,21
6,MICROSOFT CORP,MSFT,2026-11-04,6.150% Notes due 2046,6.150% Notes due 2046,21
7,MICROSOFT CORP,MSFT,2026-11-04,6.250% Notes due 2056,6.250% Notes due 2056,21
8,MICROSOFT CORP,MSFT,2026-11-04,6.350% Notes due 2066,6.350% Notes due 2066,21
11,AMAZON COM INC,AMZN,2026-10-29,2.800% Notes due 2028,2.800% notes due 2028,21
12,AMAZON COM INC,AMZN,2026-10-29,3.100% Notes due 2030,3.100% notes due 2030,21
13,AMAZON COM INC,AMZN,2026-10-29,3.350% Notes due 2032,3.350% notes due 2032,21


Remaining candidates: 41


In [24]:
os.makedirs("data", exist_ok=True)

In [25]:
print("Data folder exists:", os.path.exists("data"))

Data folder exists: True


In [26]:
print("Valid product candidates:", len(final_df))
display(final_df)

Valid product candidates: 41


,company_name,stock_name,filing_time,new_product,product_description,description_length
2,MICROSOFT CORP,MSFT,2026-11-04,5.000% Notes due 2029,5.000% Notes due 2029,21
3,MICROSOFT CORP,MSFT,2026-11-04,5.200% Notes due 2031,5.200% Notes due 2031,21
4,MICROSOFT CORP,MSFT,2026-11-04,5.350% Notes due 2033,5.350% Notes due 2033,21
5,MICROSOFT CORP,MSFT,2026-11-04,5.550% Notes due 2036,5.550% Notes due 2036,21
6,MICROSOFT CORP,MSFT,2026-11-04,6.150% Notes due 2046,6.150% Notes due 2046,21
7,MICROSOFT CORP,MSFT,2026-11-04,6.250% Notes due 2056,6.250% Notes due 2056,21
8,MICROSOFT CORP,MSFT,2026-11-04,6.350% Notes due 2066,6.350% Notes due 2066,21
11,AMAZON COM INC,AMZN,2026-10-29,2.800% Notes due 2028,2.800% notes due 2028,21
12,AMAZON COM INC,AMZN,2026-10-29,3.100% Notes due 2030,3.100% notes due 2030,21
13,AMAZON COM INC,AMZN,2026-10-29,3.350% Notes due 2032,3.350% notes due 2032,21


In [27]:
#Checking for dupes
print(
    "Duplicate products:",
    final_df.duplicated(
        subset=["company_name", "new_product"]
    ).sum()
)

Duplicate products: 7


In [28]:
# Save final cleaned results

final_df.to_csv("data/extracted_products.csv", index=False)

print("Final products:", len(final_df))
print("Final CSV saved successfully!")

Final products: 41
Final CSV saved successfully!


In [2]:
import os

print("Current folder:", os.getcwd())

csv_path = os.path.abspath("data/extracted_products.csv")

print("CSV location:", csv_path)
print("CSV exists:", os.path.isfile(csv_path))

Current folder: C:\Users\samar\PycharmProjects\JupyterProject
CSV location: C:\Users\samar\PycharmProjects\JupyterProject\data\extracted_products.csv
CSV exists: True
